# Spatial Prisoner's Dilemma: The Evolution of Cooperation

Minimax (Module 5) assumes a **zero-sum** world: my gain is exactly your loss. Most real interactions are not like that. Two drivers merging, two countries trading, two cells in a body, two strangers deciding whether to trust each other can *both* do well or *both* do badly. These are **non-zero-sum** games.

The sharpest example is the **Prisoner's Dilemma**: each side may Cooperate or Defect. Mutual cooperation pays well; mutual defection pays badly; but betraying a cooperator pays best of all. So selfish logic says defect, even though everyone defecting is worse for everyone.

This notebook watches cooperation appear anyway. We put simple strategies on a grid, let each cell play its neighbors and copy whoever is doing best, and watch global structure emerge from the local rule:

> Cooperators that cluster can defend each other; defectors that win at first then starve.


## 1. Zero-Sum vs Non-Zero-Sum

In a zero-sum game the payoffs of the two players always sum to the same total, so one player's advantage is the other's loss. Minimax is built for exactly that world.

The Prisoner's Dilemma breaks the assumption. Write `C` for cooperate and `D` for defect, and give each *individual* player a payoff:

- both cooperate: each gets the **reward** `R`
- both defect: each gets the **punishment** `P`
- you defect while they cooperate: you get the **temptation** `T`, they get the **sucker** payoff `S`

The dilemma is the ordering `T > R > P > S` (with `2R > T + S` so that alternating exploitation is not better than mutual cooperation). Because `T > R` and `P > S`, defection is the better move *whatever the opponent does* in a single game. Yet `R > P`, so two defectors both regret it.


## 2. The Payoff Matrix


**Imports and setup.** Load arrays, plotting, animation, and color tools.


In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib import animation
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch


**Payoffs.** Encode the classic Axelrod numbers `T=5 > R=3 > P=1 > S=0` as a lookup from `(my move, their move)` to my score.


In [ ]:
COOPERATE, DEFECT = 1, 0

T, R, P, S = 5, 3, 1, 0  # Temptation, Reward, Punishment, Sucker

PAYOFF = {
    (COOPERATE, COOPERATE): R,
    (COOPERATE, DEFECT): S,
    (DEFECT, COOPERATE): T,
    (DEFECT, DEFECT): P,
}


def payoff(my_move: int, their_move: int) -> int:
    return PAYOFF[(my_move, their_move)]


print("both cooperate -> ", payoff(COOPERATE, COOPERATE), "(reward)")
print("I defect, they cooperate -> ", payoff(DEFECT, COOPERATE), "(temptation: the best single round)")
print("both defect -> ", payoff(DEFECT, DEFECT), "(punishment)")
print("I cooperate, they defect -> ", payoff(COOPERATE, DEFECT), "(sucker)")


## 3. Three Strategies

A strategy decides this round's move from the opponent's **previous** move (`None` on the first round of a pairing). That tiny amount of memory is enough for the three classic players:

- **Always Defect** never cooperates.
- **Always Cooperate** never defects.
- **Tit-for-Tat** cooperates first, then copies whatever the opponent did last time.


**Strategy functions.** Each takes the opponent's last move and returns this round's move.


In [ ]:
def always_defect(opponent_last):
    return DEFECT


def always_cooperate(opponent_last):
    return COOPERATE


def tit_for_tat(opponent_last):
    return COOPERATE if opponent_last is None else opponent_last


STRATEGIES = [always_defect, always_cooperate, tit_for_tat]
STRATEGY_NAMES = ["Always Defect", "Always Cooperate", "Tit-for-Tat"]
ALLD, ALLC, TFT = 0, 1, 2


## 4. Axelrod's Tournament: Repetition Changes Everything

In **one** game, defection always wins. Robert Axelrod's 1984 insight was that real relationships are **repeated**, and the shadow of future rounds rewards reciprocity. He invited researchers to submit strategies; the simple Tit-for-Tat won.

First, a single pairing played for several rounds.


**Play a pairing.** Run two strategies against each other for `rounds` turns and return both total scores.


In [ ]:
def play_pairing(strat_a, strat_b, rounds: int):
    a_score = b_score = 0
    a_last = b_last = None
    for _ in range(rounds):
        a_move = strat_a(b_last)
        b_move = strat_b(a_last)
        a_score += payoff(a_move, b_move)
        b_score += payoff(b_move, a_move)
        a_last, b_last = a_move, b_move
    return a_score, b_score


# Tit-for-Tat refuses to be exploited twice by a defector:
print("Tit-for-Tat vs Always Defect (10 rounds):", play_pairing(tit_for_tat, always_defect, 10))
print("Tit-for-Tat vs Tit-for-Tat (10 rounds):  ", play_pairing(tit_for_tat, tit_for_tat, 10))


**Round-robin.** Let every strategy play every strategy (including its own twin) and total the scores.


In [ ]:
def round_robin(rounds: int = 10):
    totals = [0] * len(STRATEGIES)
    for i in range(len(STRATEGIES)):
        for j in range(len(STRATEGIES)):
            a, _ = play_pairing(STRATEGIES[i], STRATEGIES[j], rounds)
            totals[i] += a
    return totals


for name, total in sorted(zip(STRATEGY_NAMES, round_robin(10)), key=lambda pair: -pair[1]):
    print(f"{name:18s} total = {total}")


In this small mixed pool **Always Defect actually scores highest**, because it feasts on Always Cooperate. That is the worry. But a tournament is static: nobody learns, nobody reproduces, and everyone meets everyone equally.

Two changes flip the result, and both are forms of structure:

1. **Reproduction** - successful strategies get copied.
2. **Locality** - you mostly interact with neighbors, not the whole world.

Put both on a grid and watch.


## 5. Going Spatial

Now every cell of a grid holds one strategy. Each generation has two steps:

1. **Play.** Every cell plays an iterated game against each of its eight neighbors and sums its score.
2. **Imitate.** Every cell copies the strategy of the highest-scoring cell in its neighborhood (itself included).

This is the same shape as Conway's Game of Life or Schelling's grid: a single local rule applied everywhere, updated synchronously, producing global structure no one designed.


**Configuration.** Keep the model knobs in one named object.


In [ ]:
@dataclass(frozen=True)
class SpatialPDConfig:
    size: int = 30
    rounds: int = 10          # iterated rounds per neighbor pairing
    generations: int = 24
    seed: int = 7
    mix: tuple = (0.34, 0.33, 0.33)  # (Always Defect, Always Cooperate, Tit-for-Tat)


def initialize_grid(config: SpatialPDConfig) -> np.ndarray:
    rng = np.random.default_rng(config.seed)
    return rng.choice(3, size=(config.size, config.size), p=list(config.mix))


config = SpatialPDConfig()
grid = initialize_grid(config)
np.unique(grid, return_counts=True)


## 6. One Generation: Play Neighbors, Copy the Best

The iterated score between two strategy *types* never changes, so we compute it once into a 3x3 table. After that, a cell's fitness is just a sum of table lookups over its neighbors.


**Neighbors.** Use the eight Moore-neighborhood offsets with wraparound edges, so every cell has the same geometry.


In [ ]:
NEIGHBOR_OFFSETS = [
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1),           (0, 1),
    (1, -1),  (1, 0),  (1, 1),
]


def neighbors(row: int, col: int, size: int):
    return [((row + dr) % size, (col + dc) % size) for dr, dc in NEIGHBOR_OFFSETS]


**Pairing table.** Precompute the score each strategy earns against each strategy for `rounds` iterated turns.


In [ ]:
def pair_score_matrix(rounds: int) -> np.ndarray:
    matrix = np.zeros((3, 3))
    for i in range(3):
        for j in range(3):
            my_score, _ = play_pairing(STRATEGIES[i], STRATEGIES[j], rounds)
            matrix[i, j] = my_score
    return matrix


PAIR = pair_score_matrix(config.rounds)
print("rows = my strategy, cols = opponent strategy  [AllD, AllC, TfT]")
print(PAIR.astype(int))


**Fitness and update.** Score every cell against its neighbors, then have each cell adopt the best-scoring strategy in its closed neighborhood.


In [ ]:
def fitness_grid(grid: np.ndarray, pair: np.ndarray) -> np.ndarray:
    size = grid.shape[0]
    fitness = np.zeros_like(grid, dtype=float)
    for row in range(size):
        for col in range(size):
            strat = grid[row, col]
            fitness[row, col] = sum(pair[strat, grid[nr, nc]] for nr, nc in neighbors(row, col, size))
    return fitness


def step(grid: np.ndarray, pair: np.ndarray) -> np.ndarray:
    size = grid.shape[0]
    fitness = fitness_grid(grid, pair)
    updated = grid.copy()
    for row in range(size):
        for col in range(size):
            best_strat = grid[row, col]
            best_fit = fitness[row, col]
            for nr, nc in neighbors(row, col, size):
                if fitness[nr, nc] > best_fit:
                    best_fit = fitness[nr, nc]
                    best_strat = grid[nr, nc]
            updated[row, col] = best_strat
    return updated


## 7. Run and Watch the Shields Form

Each generation we record the board and the fraction playing each strategy.


**Simulation runner.** Capture a replayable history of grids and population fractions.


In [ ]:
def strategy_fractions(grid: np.ndarray) -> np.ndarray:
    return np.bincount(grid.ravel(), minlength=3) / grid.size


def run_spatial(config: SpatialPDConfig) -> list[dict]:
    pair = pair_score_matrix(config.rounds)
    grid = initialize_grid(config)
    history = []
    for generation in range(config.generations + 1):
        history.append({
            "generation": generation,
            "grid": grid.copy(),
            "fractions": strategy_fractions(grid),
        })
        grid = step(grid, pair)
    return history


history = run_spatial(config)
print("start [D, C, T] =", history[0]["fractions"].round(2))
print("end   [D, C, T] =", history[-1]["fractions"].round(2))


**Draw a board.** Red defects, blue always cooperates, green plays Tit-for-Tat.


In [ ]:
PD_CMAP = ListedColormap(["#ef4444", "#3b82f6", "#22c55e"])
PD_LEGEND = [
    Patch(color="#ef4444", label="Always Defect"),
    Patch(color="#3b82f6", label="Always Cooperate"),
    Patch(color="#22c55e", label="Tit-for-Tat"),
]


def draw_grid(grid: np.ndarray, ax=None, title: str | None = None):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(grid, cmap=PD_CMAP, vmin=0, vmax=2, interpolation="nearest")
    ax.set_xticks([])
    ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax


fig, axes = plt.subplots(1, 2, figsize=(10, 5))
draw_grid(history[0]["grid"], ax=axes[0], title="generation 0 (random mix)")
draw_grid(history[-1]["grid"], ax=axes[-1], title=f"generation {history[-1]['generation']} (clusters)")
axes[1].legend(handles=PD_LEGEND, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.tight_layout()
plt.show()


## 8. Tracking the Population

The picture tells one story; the population curve tells the other. Watch defection spike early as it eats naive cooperators, then collapse as it runs out of victims and meets disciplined Tit-for-Tat clusters.


**Population trace.** Plot the fraction of each strategy across generations.


In [ ]:
def plot_fractions(history: list[dict]):
    generations = [step_data["generation"] for step_data in history]
    fractions = np.array([step_data["fractions"] for step_data in history])
    colors = {"Always Defect": "#ef4444", "Always Cooperate": "#3b82f6", "Tit-for-Tat": "#22c55e"}

    fig, ax = plt.subplots(figsize=(8, 4))
    for index, name in enumerate(STRATEGY_NAMES):
        ax.plot(generations, fractions[:, index], marker="o", label=name, color=colors[name])
    ax.set_xlabel("generation")
    ax.set_ylabel("fraction of population")
    ax.set_ylim(0, 1.02)
    ax.legend()
    ax.grid(alpha=0.25)
    plt.show()


plot_fractions(history)


**Animation.** Replay the grid as local imitation reshapes the global pattern.


In [ ]:
def animate_spatial(history: list[dict]) -> HTML:
    fig, ax = plt.subplots(figsize=(5.4, 5))

    def update(frame_index: int):
        ax.clear()
        step_data = history[frame_index]
        fr = step_data["fractions"]
        draw_grid(
            step_data["grid"],
            ax=ax,
            title=f"gen {step_data['generation']} | D={fr[0]:.2f} C={fr[1]:.2f} T={fr[2]:.2f}",
        )

    anim = animation.FuncAnimation(fig, update, frames=len(history), interval=240)
    plt.close(fig)
    return HTML(anim.to_jshtml())


animate_spatial(history)


## 9. Failure Mode: Remove the Shadow of the Future

Cooperation here rests on one assumption: the game is **repeated**. Tit-for-Tat only works because it can punish a defector *next* round. Take that away by setting `rounds = 1` - a one-shot game - and Tit-for-Tat has no second move to retaliate with, so it behaves exactly like Always Cooperate.

Watch defection take the entire board.


**One-shot world.** Re-run with a single round per pairing and compare the ending.


In [ ]:
one_shot = SpatialPDConfig(rounds=1)
one_shot_history = run_spatial(one_shot)

print("iterated (rounds=10) end [D, C, T] =", history[-1]["fractions"].round(2))
print("one-shot (rounds=1)  end [D, C, T] =", one_shot_history[-1]["fractions"].round(2))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
draw_grid(history[-1]["grid"], ax=axes[0], title="repeated game: cooperation survives")
draw_grid(one_shot_history[-1]["grid"], ax=axes[1], title="one-shot game: defection wins")
plt.tight_layout()
plt.show()


## 10. Mini-Challenge

Try one:

1. Find the smallest starting fraction of Tit-for-Tat that still survives to the final generation for a fixed seed.
2. Add a `random` strategy (cooperate with probability `p`) and see whether it ever forms clusters.
3. Replace "copy the best neighbor" with "copy a neighbor with probability proportional to its score" and compare the patterns.
4. Drop locality instead of repetition: let each cell play eight *random* cells and imitate a random one. Does repetition alone still protect cooperation?
5. Lower `T` toward `R`. At what temptation does cooperation stop needing the shield?


**Invariant check.** Confirm the update is synchronous: the next grid is a function only of the current grid's fitness field.


In [ ]:
pair = pair_score_matrix(config.rounds)
once = step(history[0]["grid"], pair)
twice = step(history[0]["grid"], pair)
print("two updates of the same board agree:", bool(np.array_equal(once, twice)))
print("update matches the recorded next generation:", bool(np.array_equal(once, history[1]["grid"])))


## Visual Trace + Rigor Studio

**Problem frame.** Show how cooperation can survive among self-interested agents when the game is repeated and successful strategies are imitated locally.

**Interactive animation target.** Animate the strategy grid (red Defect, blue Cooperate, green Tit-for-Tat) across generations beside the population-fraction curve, with controls for the starting mix, iterated rounds, and seed.

**Correctness handle.** The update is synchronous: each cell's next strategy is the highest-scoring strategy in its closed neighborhood under the current fitness field, so a strategy can only spread into a cell whose best neighbor already plays it.

**Complexity handle.** Each generation is `O(generations * cells * neighbors)` grid work; the iterated pairing scores are precomputed once into a 3x3 table at `O(strategies^2 * rounds)`.

**Failure mode to test.** Set `rounds = 1`. Without the shadow of the future, Tit-for-Tat cannot retaliate and behaves like Always Cooperate, so defection dominates the whole board. Repetition is the assumption that makes reciprocity possible.

**Studio task.** Run the iterated model, find the generation where defectors peak, then explain why Tit-for-Tat clusters stop shrinking and begin to hold their boundary while isolated cooperators are eaten.

**Transfer.** Minimax (Module 5) is the zero-sum cousin of this game: one optimal opponent, my gain is your loss. Here the game is non-zero-sum and the "winner" is not searched for but emerges from local play, the way Schelling clustering and Game of Life structure emerge from local rules.


## Sources and Further Reading

- Robert Axelrod and William D. Hamilton, [The Evolution of Cooperation](https://www.science.org/doi/10.1126/science.7466396), *Science* 211, 1390-1396 (1981)
- Robert Axelrod, *The Evolution of Cooperation* (1984) - the tournament and Tit-for-Tat
- Martin A. Nowak and Robert M. May, [Evolutionary games and spatial chaos](https://www.nature.com/articles/359826a0), *Nature* 359, 826-829 (1992) - the spatial model

This is a compact teaching version. Real evolution of cooperation also involves reputation, kinship, punishment, noise, and many more strategies than three.
